In [4]:
# -----------------------------
# Append MarketCap from Valuation sheet to existing firm CSVs
# -----------------------------
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR.parent / "data"
XLSX_DIR = DATA_DIR / "accounting_sheets"
CSV_DIR = DATA_DIR / "prof_components_extracted"

EXCHANGES = ["cox", "hex", "isx", "obx", "stx"]


In [5]:


def clean_label(x):
    """Normalize sheet labels for matching."""
    if x is None:
        return ""
    s = str(x).strip()
    s = " ".join(s.split())
    return s.lower()


def to_number(x):
    """Convert Excel cell values like '403,9', '1 234,5', etc. to float."""
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    s = str(x).strip()
    if s == "":
        return np.nan

    # Handle unicode minus, spaces, percent, NBSP
    s = s.replace("\u2212", "-").replace("%", "").replace("\xa0", "").replace(" ", "")

    # Common European decimal format
    # If both comma and dot exist, assume dot is thousands separator if comma is decimal
    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        s = s.replace(".", "").replace(",", ".")
    else:
        s = s.replace(",", "")

    try:
        return float(s)
    except Exception:
        return np.nan


def extract_market_cap_from_valuation(xlsx_path: Path) -> pd.DataFrame:
    """
    Return DataFrame with columns: Year, MarketCap
    from the 'Valuation' sheet.
    Picks the row named 'Market Capitalization' that actually has values,
    not the header/section row and not the 5-year average row.
    """
    wb = load_workbook(xlsx_path, data_only=True, read_only=True)

    if "Valuation" not in wb.sheetnames:
        return pd.DataFrame(columns=["Year", "MarketCap"])

    ws = wb["Valuation"]

    # Read entire sheet into a simple list-of-lists
    rows = list(ws.iter_rows(values_only=True))
    if not rows:
        return pd.DataFrame(columns=["Year", "MarketCap"])

    # Find the year header row: row containing several 4-digit years
    year_row_idx = None
    year_cols = []

    for i, row in enumerate(rows):
        candidate_cols = []
        for j, val in enumerate(row):
            if val is None:
                continue
            sval = str(val).strip()
            if sval.isdigit() and len(sval) == 4:
                yr = int(sval)
                if 1900 <= yr <= 2100:
                    candidate_cols.append(j)

        if len(candidate_cols) >= 2:
            year_row_idx = i
            year_cols = candidate_cols
            break

    if year_row_idx is None or not year_cols:
        return pd.DataFrame(columns=["Year", "MarketCap"])

    years = {}
    for j in year_cols:
        val = rows[year_row_idx][j]
        try:
            years[j] = int(str(val).strip())
        except Exception:
            pass

    if not years:
        return pd.DataFrame(columns=["Year", "MarketCap"])

    # Find the correct Market Capitalization row:
    # exact label match, not 5-year average, and must contain numeric values in year columns
    chosen_row = None
    for i, row in enumerate(rows):
        label = clean_label(row[0] if len(row) > 0 else "")
        if label != "market capitalization":
            continue

        numeric_count = 0
        vals = []
        for j in years.keys():
            val = row[j] if j < len(row) else None
            num = to_number(val)
            vals.append(num)
            if pd.notna(num):
                numeric_count += 1

        if numeric_count >= 1:
            chosen_row = row
            break

    if chosen_row is None:
        return pd.DataFrame(columns=["Year", "MarketCap"])

    out = []
    for j, yr in years.items():
        val = chosen_row[j] if j < len(chosen_row) else None
        num = to_number(val)
        if pd.notna(num):
            out.append({"Year": int(yr), "MarketCap": float(num)})

    return pd.DataFrame(out).sort_values("Year").reset_index(drop=True)



In [6]:

n_updated = 0
n_missing_csv = 0
n_missing_sheet_or_data = 0
n_missing_xlsx = 0
problems = []

for exch in EXCHANGES:
    xlsx_dir = XLSX_DIR / f"{exch}_xlsx"
    if not xlsx_dir.exists():
        print(f"Skipping missing folder: {xlsx_dir}")
        continue

    xlsx_files = sorted(xlsx_dir.glob("*.xlsx"))
    print(f"\nExchange {exch.upper()}: found {len(xlsx_files)} xlsx files")

    for xlsx_path in xlsx_files:
        ticker = xlsx_path.stem
        csv_path = CSV_DIR / f"{ticker}.csv"

        if not csv_path.exists():
            n_missing_csv += 1
            continue

        try:
            mc_df = extract_market_cap_from_valuation(xlsx_path)

            if mc_df.empty:
                n_missing_sheet_or_data += 1
                problems.append((ticker, "No Valuation sheet or no Market Capitalization values found"))
                continue

            firm_df = pd.read_csv(csv_path)

            if "Year" not in firm_df.columns:
                problems.append((ticker, "CSV missing Year column"))
                continue

            firm_df["Year"] = pd.to_numeric(firm_df["Year"], errors="coerce").astype("Int64")
            mc_df["Year"] = pd.to_numeric(mc_df["Year"], errors="coerce").astype("Int64")

            # Drop existing MarketCap if present, then merge fresh values
            if "MarketCap" in firm_df.columns:
                firm_df = firm_df.drop(columns=["MarketCap"])

            firm_df = firm_df.merge(mc_df, on="Year", how="left")

            firm_df.to_csv(csv_path, index=False)
            n_updated += 1

        except Exception as e:
            problems.append((ticker, str(e)))

print("\nDone.")
print(f"Updated CSVs: {n_updated}")
print(f"Missing matching CSV: {n_missing_csv}")
print(f"No Valuation sheet / no usable MarketCap row: {n_missing_sheet_or_data}")
print(f"Problems logged: {len(problems)}")

if problems:
    print("\nFirst 20 problems:")
    for t, msg in problems[:20]:
        print(f"  {t}: {msg}")


Exchange COX: found 88 xlsx files

Exchange HEX: found 123 xlsx files

Exchange ISX: found 21 xlsx files

Exchange OBX: found 157 xlsx files

Exchange STX: found 270 xlsx files

Done.
Updated CSVs: 632
Missing matching CSV: 25
No Valuation sheet / no usable MarketCap row: 2
Problems logged: 2

First 20 problems:
  ALIVsdb.ST: No Valuation sheet or no Market Capitalization values found
  ASKER.ST: No Valuation sheet or no Market Capitalization values found
